# Predicting CRISPRi knockdowns in cell lines nobody has perturbed

**Virtual Cell Challenge 2026: our final pipeline, end to end.** Helen Henry (University of Tampere), Nyle Hussain
(University of Glasgow), Kevin Henry.

**The story in one paragraph.** The challenge hands us three human cell lines, D, E and F, as about 18,400
resting ("control") cells each, and a list of 300 genes. For each gene it asks: if you switched this gene down
in this line, what would 400 of its cells look like, RNA molecule by RNA molecule? Nobody has run that
experiment on these lines, and we cannot either. But public experiments have switched many of the same genes
down in *other* cell lines. Our pipeline borrows those measured effects, decides how much to trust each one,
moves them onto the new line, and then draws realistic cells around them. In software terms: **stage 1** predicts a
diff (how much each gene's level changes), **stage 2** renders that diff into concrete samples, and the
grader runs a statistical test on the samples, so the rendering matters as much as the diff.

This notebook is the pipeline that built our final submission, not a reconstruction of it: run in FULL mode,
it rebuilds that file byte for byte from the public data and the challenge's controls.

## How to read this notebook

- **The story is in sections 1 to 15**, in four parts: understand the problem (1, 2), look at the data (3 to 7),
  build the pipeline one measured piece at a time (8 to 12), and the final file (13 to 15). Every section has
  the same shape: **the question**, **the experiment** (code you can rerun), **the figure**, **what we learned**
  (with the number the run printed) and **the playbook lesson**, a rule that holds beyond this challenge. The
  lessons are collected as a checklist at the end.
- **The code is in the package `vccpipe`** next to this notebook (eleven modules, each a readable file), so the
  story starts right after the setup. Each section shows the function it teaches, and the sections explain the
  ideas as they meet them.
- **Every number and figure is computed by this notebook** from the public data and the challenge's controls.
  Scores of earlier versions on the challenge's leaderboard are quoted from our records and marked as quoted.
- **Colours mean the same thing everywhere**: line D blue, E orange, F aqua; the public training screens by
  tissue (blood violet, stem cells green, liver yellow, epithelial magenta); our method in near-black against a
  grey alternative; changes blue for down, red for up.
- **Run modes.** `FAST` (the default) runs the exploration on small public downloads, under an hour on a laptop.
  `FULL` rebuilds the final file (hours; ~80 GB of memory). The official scorer's statistics need a CUDA GPU:
  the sections that score (2, 8, 10, 15) say so and are skipped without one.

## Words we use

| word | meaning |
|---|---|
| **cell line** | a population of human cells grown in the lab from one donor or one tumour, so its cells are near-copies of each other |
| **gene expression, counts** | how many RNA molecules of each gene were captured from one cell: a cell is a vector of 18,533 whole numbers |
| **control cells** | cells given a guide that targets nothing ("non-targeting"): the unperturbed baseline |
| **knockdown, CRISPRi** | switching one gene's expression down: a guide RNA steers a disabled Cas9 protein, fused to a repressor, onto the gene's promoter (the DNA stretch where the gene's reading starts); the DNA itself is not cut |
| **screen, Perturb-seq** | an experiment that knocks down many genes, one per cell, and reads every cell's RNA |
| **log2 fold change** | log2 of (level after the knockdown / level in the controls): +1 doubled, -1 halved, 0 unchanged |
| **profile** | one knockdown's log2 fold change over all genes: the "diff" stage 1 predicts |
| **CPM** | counts per million: a gene's share of the counts, times a million (a level that does not depend on how deeply a cell was read) |
| **UMI, depth** | a molecule's barcode; a cell's depth is how many RNA molecules (UMIs) were counted in it |
| **pseudobulk** | many cells' counts added up into one profile |
| **Wilcoxon rank-sum test** | asks whether one group's values tend to rank above another's; the scorer runs it gene by gene, 400 predicted cells against the line's real controls |
| **called, significant** | a gene whose test p-value, corrected for testing thousands of genes at once (Benjamini-Hochberg), is below 0.05 |
| **10x Flex** | the lab method that measured the challenge's cells; screens measured with other methods see genes somewhat differently |
| **tissue, lineage** | the kind of cell a line comes from: blood, liver, brain, skin lining (epithelial) |
| **pluripotent stem cell (H1, iPSC)** | a cell that can still become any tissue; H1 is an embryonic stem-cell line, KOLF an induced one (iPSC) |
| **p53** | a protein that stops a stressed cell from growing; many cancer lines have lost it, and a line that has it answers some knockdowns differently |

## 0. Setup

`VCC_MODE`: `FAST` (default) runs the exploration on small public downloads; `FULL` rebuilds every reference table
and the final submission (hours, a machine with ~80 GB of memory). Paths: `VCC_NB_DATA` (default `data`) holds the
tables, the challenge's controls, the working files and the saved official scores (`data/scores`).

**Reproducibility is a feature you build first.** The final file is rebuilt here byte for byte, which needs three
things pinned: the library versions, the order of every floating-point sum, and the number of threads the linear
algebra library (BLAS) uses, because a matrix product split over more threads adds its numbers in a different order
and moves the last digit. The thread count is therefore fixed before numpy loads, and each step sets its own.

In [1]:
import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "6")
os.environ.setdefault("OMP_NUM_THREADS", "6")
os.environ.setdefault("MKL_NUM_THREADS", "6")
os.environ.setdefault("NUMPY_MADVISE_HUGEPAGE", "0")
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")  # library deprecation and empty-slice notices; every check below raises on failure

MODE = os.environ.get("VCC_MODE", "FAST").upper()
assert MODE in ("FAST", "FULL"), MODE
DATA = Path(os.environ.get("VCC_NB_DATA", "data"))
TABLES = Path(os.environ.get("VCC_TABLES", DATA / "tables"))  # the public screens as tables
PHASE = Path(os.environ.get("VCC_PHASE_DIR", DATA / "controls_final"))  # the challenge's final controls
WORK = Path(os.environ.get("VCC_WORK", DATA / "work"))
SUBMIT = os.environ.get("VCC_SUBMIT", "") == "yes"  # never on by default


def gpu_available() -> bool:
    """The official scorer's DE engine (gpudge) needs a CUDA GPU; without one cell-eval2 falls back to another engine
    whose numbers differ. Cells that report the scorer's numbers run only when a GPU is visible."""
    try:
        import torch

        return bool(torch.cuda.is_available())
    except Exception:  # noqa: BLE001
        return False


GPU = gpu_available()
print("mode", MODE, "| GPU for the scorer:", GPU)

mode FULL | GPU for the scorer: True


## The package

The pipeline's code lives in the package `vccpipe` next to this notebook: plain numpy and scipy in eleven documented
modules (unit tests in `tests/`), so the story can start right away. Each section shows the function it teaches
(`show(...)` prints its source).

| module | what it holds | used in |
|---|---|---|
| `inputs` | the public screens as tables; each knockdown's borrowed profile, the significance gate, the neighbour prior | 4, 10, 13 |
| `constants`, `levels` | numbers fitted once on the training screens; the rank test's power from a line's own controls | 11, 13 |
| `combiner` | E, the learned combiner (a small ridge regression across cell types) | 10, 13 |
| `generator` | stage 2: from a predicted change to 400 cells of whole-number counts | 9, 11, 13 |
| `stage1` | stage 1's pieces, ending in each knockdown's targets | 10, 13 |
| `phase` | the entry points: `prepare_phase`, `probe_phase`, `predict_context`, `build_phase` | 13, 15 |
| `harness` | leave one line out, scored by the official scorer | 8, 10, 15 |
| `figures` | the palette and figure helpers | every figure |
| `rebuild`, `fast` | rebuilding the reference tables from public downloads; FAST mode's small inputs | appendix; FAST |

In [2]:
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm

from vccpipe import *  # the package's functions, classes and constants, and `show`

## FAST mode's data

In FAST mode the sections below read a small stand-in for the reference tables, built here from public files:
2,000 controls and 100 cells of each of 60 knockdowns per essential-gene line (K562, RPE1, HepG2, Jurkat), read
by byte range; the first 10,000 cells of the 2025 challenge's H1 validation split (FULL uses its training split);
the K562 genome-wide pseudobulk, the pathway screens and DepMap: about 3.5 GB in all. The numbers therefore
differ from a FULL run's; each one printed is computed from these data. KOLF, the other screens and the final
build are FULL only, and section 7's test needs FULL's 18,400 controls: with FAST's 2,000, even its planted
fluctuation is barely found. The reads are polite to the public servers: at most four connections to one
server, and a growing wait (or the server's own Retry-After) whenever one says it is busy. What was read is
kept under the downloads folder, so a rerun reads nothing twice.

In [3]:
if MODE == "FAST":
    TABLES = DATA / "fast"
    if not (TABLES / "phi.npz").exists():
        build_fast_tables(TABLES, Path(os.environ.get("VCC_DOWNLOADS", DATA / "downloads")), PHASE)
    print("FAST tables:", sorted(p.name for p in TABLES.iterdir()))